In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.retail_lakehouse")

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show()

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.retail_lakehouse.landing")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/retail_lakehouse/landing/retail_day_01"))

In [0]:
landing = "/Volumes/workspace/retail_lakehouse/landing/retail_day_01"
schema  = "workspace.retail_lakehouse"

# sanity check: can Spark see the files?
display(dbutils.fs.ls(landing))

In [0]:
customers_df = (
    spark.read
    .option("header", "true")       # first row is column names
    .option("inferSchema", "true")  # let Spark guess types from the data
    .csv(f"{landing}/customers.csv")
)

customers_df.printSchema()   # look at the columns + guessed types
customers_df.show(5)

In [0]:
from pyspark.sql import functions as F

customers_df = (
    customers_df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("customers.csv"))
)

In [0]:
(
    customers_df.write
    .format("delta")
    .mode("overwrite")              # replace the table each run (fine for Bronze rebuilds)
    .saveAsTable(f"{schema}.bronze_customers")
)

In [0]:
for name in ["products", "orders"]:
    df = (spark.read
          .option("header", "true").option("inferSchema", "true")
          .csv(f"{landing}/{name}.csv")
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.lit(f"{name}.csv")))
    (df.write.format("delta").mode("overwrite")
       .saveAsTable(f"{schema}.bronze_{name}"))
    print(f"wrote bronze_{name}: {df.count()} rows")

In [0]:
from pyspark.sql.functions import col, count, when

customers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customers_df.columns
]).show()

In [0]:
customers_df = customers_df.dropDuplicates(["customer_id"])

In [0]:
customers_df.dtypes

In [0]:
products_df = spark.read.table(f"{schema}.bronze_products")

In [0]:
# Assuming your DataFrame is named products_df
from pyspark.sql import Window
from pyspark.sql import functions as F

# 1. Define the columns that define a duplicate (e.g., product_id)
# If you want to check the entire row, use products_df.columns instead
duplicate_columns = ["product_id"] 

# 2. Define the window. An orderBy is required for row_number()
# You can order by a timestamp, an ID, or literal(1) if order doesn't matter
window_spec = Window.partitionBy(duplicate_columns).orderBy(F.lit(1))

# 3. Create the boolean mask column (True for duplicates, False for unique/first occurrence)
products_df_with_mask = products_df.withColumn(
    "is_duplicate", 
    F.row_number().over(window_spec) > 1
)

# Show the results
products_df_with_mask.show()


In [0]:
products_df = products_df.dropDuplicates(["product_id"])

In [0]:
from pyspark.sql.functions import col, count, when

products_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in products_df.columns
]).show()

In [0]:
products_df.dtypes

In [0]:
orders_df = spark.read.table(f"{schema}.bronze_orders")

In [0]:
# Assuming your DataFrame is named products_df
from pyspark.sql import Window
from pyspark.sql import functions as F

# 1. Define the columns that define a duplicate (e.g., product_id)
# If you want to check the entire row, use products_df.columns instead
duplicate_columns = ["order_id"] 

# 2. Define the window. An orderBy is required for row_number()
# You can order by a timestamp, an ID, or literal(1) if order doesn't matter
window_spec = Window.partitionBy(duplicate_columns).orderBy(F.lit(1))

# 3. Create the boolean mask column (True for duplicates, False for unique/first occurrence)
orders_df_with_mask = orders_df.withColumn(
    "is_duplicate", 
    F.row_number().over(window_spec) > 1
)

# Show the results
orders_df_with_mask.show()


In [0]:
from pyspark.sql.functions import col, count, when

orders_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders_df.columns
]).show()

In [0]:
orders_df.dtypes

In [0]:
from pyspark.sql.functions import to_timestamp, to_date, col
orders_df = (
    orders_df
    .withColumn("order_date", to_date(col("order_ts")))       # timestamp -> just the date
)

In [0]:
dfs_to_write = {
    "silver_customers": customers_df,
    "silver_products":  products_df,
    "silver_orders":    orders_df,
}

for table_name, df in dfs_to_write.items():
    (df.write
       .format("delta")
       .mode("overwrite")
       .saveAsTable(f"workspace.retail_lakehouse.{table_name}"))
    print(f"wrote {table_name}: {df.count()} rows")

In [0]:
orders_enriched_df = (
    orders_df
    .join(customers_df, on="customer_id", how="left")
    .join(products_df, on="product_id", how="left")
    .select(
        orders_df["order_id"],
        orders_df["customer_id"],
        orders_df["product_id"],
        orders_df["quantity"],
        orders_df["amount"],
        orders_df["order_ts"],
        orders_df["order_date"],
        customers_df["city"],
        customers_df["state"],
        customers_df["age"],
        customers_df["gender"],
        products_df["category"],
        products_df["brand"],
        products_df["price"],
    )
)

In [0]:
(orders_enriched_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.silver_orders_enriched"))

print(f"wrote silver_orders_enriched: {orders_enriched_df.count()} rows")

In [0]:
spark.sql("SHOW TABLES IN workspace.retail_lakehouse").show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# read from the table, not memory — same habit as the join step
orders_enriched_df = spark.table("workspace.retail_lakehouse.silver_orders_enriched")

w = Window.partitionBy("customer_id").orderBy(col("order_ts").desc())

latest_purchase_df = (
    orders_enriched_df
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

(latest_purchase_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.silver_customer_latest_purchase"))

print(f"wrote silver_customer_latest_purchase: {latest_purchase_df.count()} rows")

In [0]:
spark.sql("SHOW TABLES IN workspace.retail_lakehouse").show()

In [0]:
for t in ["silver_customers", "silver_products", "silver_orders",
          "silver_orders_enriched", "silver_customer_latest_purchase"]:
    n = spark.table(f"workspace.retail_lakehouse.{t}").count()
    print(f"{t}: {n}")